# Exposure-Aware Air Quality Health Risk Classification

This notebook presents an end-to-end machine learning pipeline to classify
air-pollution health risk levels based on prolonged exposure patterns rather
than raw AQI values.


In [1]:
# Install scikit-learn inside the notebook environment
import sys
!{sys.executable} -m pip install scikit-learn



Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: C:\Users\ayazp\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## 1. Data Loading & Initial Audit

In this section, we load the dataset and perform an initial structural
inspection to understand columns, data types, and missing values.


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
df = pd.read_csv("../data/city_day.csv")
df.shape
df.head()
df.info()
print(df.columns)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29531 entries, 0 to 29530
Data columns (total 16 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   City        29531 non-null  object 
 1   Date        29531 non-null  object 
 2   PM2.5       24933 non-null  float64
 3   PM10        18391 non-null  float64
 4   NO          25949 non-null  float64
 5   NO2         25946 non-null  float64
 6   NOx         25346 non-null  float64
 7   NH3         19203 non-null  float64
 8   CO          27472 non-null  float64
 9   SO2         25677 non-null  float64
 10  O3          25509 non-null  float64
 11  Benzene     23908 non-null  float64
 12  Toluene     21490 non-null  float64
 13  Xylene      11422 non-null  float64
 14  AQI         24850 non-null  float64
 15  AQI_Bucket  24850 non-null  object 
dtypes: float64(13), object(3)
memory usage: 3.6+ MB
Index(['City', 'Date', 'PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2',
       'O3', 'Benzen

In [4]:
missing_pct = (df.isna().mean()*100).sort_values(ascending=False)
print(missing_pct)

Xylene        61.322001
PM10          37.723071
NH3           34.973418
Toluene       27.229014
Benzene       19.041008
AQI           15.851139
AQI_Bucket    15.851139
PM2.5         15.570079
NOx           14.171549
O3            13.619586
SO2           13.050692
NO2           12.139785
NO            12.129626
CO             6.972334
Date           0.000000
City           0.000000
dtype: float64


## 2. Data Cleaning & Preprocessing

This section handles missing values, removes unreliable cities,
and ensures the dataset is suitable for temporal exposure modeling.


In [5]:
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df = df.dropna(subset=['Date'])
df = df.sort_values(['City', 'Date']).reset_index(drop=True)


In [6]:
pollutant_cols = [
    'PM2.5','PM10','NO','NO2','NOx','NH3',
    'CO','SO2','O3','Benzene','Toluene','Xylene'
]

df[pollutant_cols] = df[pollutant_cols].mask(df[pollutant_cols] < 0)

In [7]:
impute_cols = ['PM2.5','PM10','NO','NO2','NOx','NH3','CO','SO2','O3']


In [8]:
df[impute_cols] = (
    df.groupby('City')[impute_cols]
    .transform(lambda x : x.fillna(x.median()))
)

C:\Users\ayazp\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\numpy\lib\_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


In [9]:
df[impute_cols].describe()

,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3
count,29531.000000,27522.000000,29531.000000,29531.000000,28362.000000,27522.000000,29531.000000,29531.000000,29369.000000
mean,64.495018,112.321444,17.586426,28.038496,32.045347,22.317420,2.469716,15.120249,33.817075
std,60.418818,77.661482,21.513692,23.404976,30.101181,22.208817,7.014473,17.676906,20.499507
min,0.040000,0.010000,0.020000,0.010000,0.000000,0.010000,0.000000,0.010000,0.010000
25%,28.080000,59.960000,6.100000,12.080000,13.980000,10.330000,0.510000,6.050000,19.890000
50%,48.170000,100.685000,10.580000,21.460000,24.290000,14.500000,0.900000,9.900000,29.620000
75%,76.345000,130.060000,22.220000,36.650000,39.510000,28.787500,1.500000,15.600000,43.160000
max,949.990000,1000.000000,390.680000,362.210000,467.630000,352.890000,175.810000,193.860000,257.730000


In [10]:
city_counts = df['City'].value_counts()
city_counts.describe()


count      26.000000
mean     1135.807692
std       706.635296
min       113.000000
25%       415.000000
50%      1113.000000
75%      1969.000000
max      2009.000000
Name: count, dtype: float64

In [11]:
city_counts = df['City'].value_counts()
valid_cities = city_counts[city_counts >= 365].index
df = df[df['City'].isin(valid_cities)].reset_index(drop=True)

In [12]:
df['City'].nunique()
df['City'].value_counts().describe()

count      20.000000
mean     1409.550000
std       559.650029
min       386.000000
25%       947.750000
50%      1341.500000
75%      2009.000000
max      2009.000000
Name: count, dtype: float64

In [13]:
df['year'] = df['Date'].dt.year
df['month'] = df['Date'].dt.month
df['dayofweek'] = df['Date'].dt.dayofweek

df['is_winter'] = df['month'].isin([11, 12, 1, 2]).astype(int)

# Cyclical temporal features
import numpy as np
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
df['dayofweek_sin'] = np.sin(2 * np.pi * df['dayofweek'] / 7)
df['dayofweek_cos'] = np.cos(2 * np.pi * df['dayofweek'] / 7)


## 3. Exploratory Data Analysis (EDA)

Exploratory analysis is performed to understand pollutant distributions,
temporal trends, and variability across cities.


In [14]:
pollutants = ['PM2.5','PM10','NO2','SO2','CO','O3']


### 3.1 Distribution of Major Pollutants

This plot examines the distribution of major pollutants to understand skewness,
outliers, and variability across observations.


In [15]:
pollutants = ['PM2.5', 'PM10', 'NO2', 'SO2', 'CO', 'O3']

plt.figure(figsize=(12,6))
for p in pollutants:
    sns.kdeplot(df[p], label=p)

plt.legend()
plt.title("Distribution of Major Air Pollutants")
plt.show()

C:\Users\ayazp\AppData\Local\Temp\ipykernel_29400\574663429.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 3.2 Correlation Between Pollutants

This heatmap shows correlations among pollutants to assess co-movement
and redundancy in pollutant measurements.


In [16]:
plt.figure(figsize=(8,6))
corr = df[['PM2.5','PM10','NO2','SO2','CO','O3']].corr()

sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Pollutant Correlation Heatmap")
plt.show()


C:\Users\ayazp\AppData\Local\Temp\ipykernel_29400\2959792271.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Feature Engineering

This section constructs exposure-aware features that capture
pollution accumulation, temporal persistence, and volatility.


In [17]:
# Find temporal split cutoff date (80% train, 20% test chronologically)
cutoff_date = df['Date'].sort_values().iloc[int(len(df) * 0.8)]
train_mask = df['Date'] < cutoff_date
test_mask = df['Date'] >= cutoff_date

from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
# Fit scaler only on the training set to prevent data leakage
scaler.fit(df.loc[train_mask, pollutants])
df[[p + '_norm' for p in pollutants]] = scaler.transform(df[pollutants])


In [18]:
df['Daily_HRS'] = (
    0.35 * df['PM2.5_norm'] +
    0.20 * df['PM10_norm'] +
    0.15 * df['NO2_norm'] +
    0.10 * df['SO2_norm'] +
    0.10 * df['CO_norm'] +
    0.10 * df['O3_norm']
)


In [19]:
df = df.sort_values(['City' , 'Date'])

In [20]:
df['HRS_3day'] = (
    df.groupby('City')['Daily_HRS']
      .rolling(3, min_periods=1)
      .mean()
      .reset_index(level=0, drop=True)
)

df['HRS_7day'] = (
    df.groupby('City')['Daily_HRS']
      .rolling(7, min_periods=1)
      .mean()
      .reset_index(level=0, drop=True)
)


### 3.3 Temporal Behavior of Health Risk Score

This plot shows how pollution exposure evolves over time for a single city,
illustrating persistence and gradual transitions.


In [21]:
sample_city = df['City'].iloc[0]
city_df = df[df['City'] == sample_city]

plt.figure(figsize=(12,5))
plt.plot(city_df['Date'], city_df['Daily_HRS'], alpha=0.4, label='Daily HRS')
plt.plot(city_df['Date'], city_df['HRS_7day'], linewidth=2, label='7-day Rolling HRS')

plt.title(f"Temporal Exposure Pattern – {sample_city}")
plt.legend()
plt.show()


C:\Users\ayazp\AppData\Local\Temp\ipykernel_29400\2502528655.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [22]:
df['HRS_lag1'] = df.groupby('City')['Daily_HRS'].shift(1)


In [23]:
df['VOC_index'] = df[['Benzene' , 'Toluene' , 'Xylene']].sum(axis=1 , min_count=1)


In [24]:
df[['Daily_HRS', 'HRS_3day', 'HRS_7day']].describe()


,Daily_HRS,HRS_3day,HRS_7day
count,26182.000000,26182.000000,26182.000000
mean,0.079886,0.079910,0.079947
std,0.045454,0.043550,0.042052
min,0.007044,0.008321,0.010917
25%,0.049965,0.050592,0.050592
50%,0.067445,0.068045,0.068748
75%,0.100522,0.100686,0.100991
max,0.588110,0.427549,0.421947


In [25]:
df = df.dropna(subset=['HRS_7day']).reset_index(drop=True)


In [26]:
# Find temporal split cutoff date (80% train, 20% test chronologically)
cutoff_date = df['Date'].sort_values().iloc[int(len(df) * 0.8)]
train_mask = df['Date'] < cutoff_date
test_mask = df['Date'] >= cutoff_date
low_th , high_th = df.loc[train_mask, 'HRS_7day'].quantile([0.33 , 0.66])
low_th , high_th


(0.05535693951246338, 0.08785437715672387)

In [27]:
def risk_label(x):
    if x <= low_th:
        return 'Low'
    elif x <= high_th:
        return 'Moderate'
    else:
        return 'High'

df['Health_Risk_Level'] = df['HRS_7day'].apply(risk_label)


### 3.4 Distribution of Health Risk Labels


In [28]:
plt.figure(figsize=(6,4))
sns.countplot(x=df['Health_Risk_Level'])
plt.title("Distribution of Health Risk Levels")
plt.show()


C:\Users\ayazp\AppData\Local\Temp\ipykernel_29400\1753754006.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [29]:
df = df.sort_values(['City', 'Date'])

df['HRS_7day_std'] = (
    df.groupby('City')['Daily_HRS']
      .transform(lambda x: x.rolling(window=7, min_periods=3).std())
)


In [30]:
features = [
    'month_sin', 'month_cos', 'dayofweek_sin', 'dayofweek_cos', 'is_winter',
    'HRS_lag1', 'HRS_7day_std',
    'VOC_index'
]

X = df[features]
y = df['Health_Risk_Level']


In [31]:
# Fit median imputer on X_train (train_mask) and apply to both splits to prevent leakage
train_medians = X.loc[train_mask].median()
X = X.fillna(train_medians)


## 6. Modeling & Evaluation

Machine learning models are trained using a time-aware split,
with emphasis on detecting high-risk exposure periods.


In [32]:
# Apply chronological split masks to X and y
X_train = X[train_mask]
y_train = y[train_mask]
X_test = X[test_mask]
y_test = y[test_mask]


In [33]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

log_reg_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(max_iter=1000))
])

log_reg_pipeline.fit(X_train, y_train)
y_pred_lr = log_reg_pipeline.predict(X_test)


In [34]:
from sklearn.ensemble import  RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=50,
    random_state=42
)

rf.fit(X_train , y_train)
y_pred_rf = rf.predict(X_test)

In [35]:
from sklearn.metrics import classification_report


print("Logistic Regression")
print(classification_report(y_test, y_pred_lr))

print("Random Forest")
print(classification_report(y_test, y_pred_rf))


Logistic Regression
              precision    recall  f1-score   support

        High       0.82      0.86      0.84      1393
         Low       0.88      0.86      0.87      1836
    Moderate       0.78      0.76      0.77      2015

    accuracy                           0.83      5244
   macro avg       0.83      0.83      0.83      5244
weighted avg       0.83      0.83      0.83      5244

Random Forest
              precision    recall  f1-score   support

        High       0.81      0.87      0.84      1393
         Low       0.90      0.84      0.87      1836
    Moderate       0.77      0.78      0.77      2015

    accuracy                           0.82      5244
   macro avg       0.83      0.83      0.83      5244
weighted avg       0.83      0.82      0.82      5244



## 7. Error Analysis & Interpretation

This section examines misclassifications to understand
model behavior near risk boundaries.


In [36]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

cm = confusion_matrix(
    y_test , y_pred_rf,
    labels=['Low' , 'Moderate' , 'High']
)

plt.figure(figsize=(6,5))
sns.heatmap(
    cm, annot=True, fmt='d',
    xticklabels=['Low','Moderate','High'],
    yticklabels=['Low','Moderate','High']
)
plt.title("Random Forest – Confusion Matrix")
plt.show()


C:\Users\ayazp\AppData\Local\Temp\ipykernel_29400\2537285074.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [37]:
importance = pd.Series(
    rf.feature_importances_,
    index=features
).sort_values(ascending=False)

importance

HRS_lag1         0.609336
HRS_7day_std     0.192800
VOC_index        0.099415
month_cos        0.031325
month_sin        0.024949
dayofweek_sin    0.017503
is_winter        0.012387
dayofweek_cos    0.012287
dtype: float64

In [38]:
error_df = X_test.copy()
error_df['true'] = y_test.values
error_df['pred'] = y_pred_rf

errors = error_df[error_df['true'] != error_df['pred']]
errors.head()

,month_sin,month_cos,dayofweek_sin,dayofweek_cos,is_winter,HRS_lag1,HRS_7day_std,VOC_index,true,pred
1808,-2.449294e-16,1.0,-0.974928,-0.222521,1,0.068118,0.027429,3.99,High,Moderate
1810,-2.449294e-16,1.0,0.000000,1.000000,1,0.063843,0.027604,3.99,High,Moderate
1813,-2.449294e-16,1.0,0.433884,-0.900969,1,0.090036,0.019415,3.99,Moderate,High
1817,-2.449294e-16,1.0,0.000000,1.000000,1,0.075820,0.020293,3.99,High,Moderate
1818,-2.449294e-16,1.0,0.781831,0.623490,1,0.078087,0.019119,3.99,High,Moderate


## 8. Conclusions

The model successfully captures exposure-based health risk patterns,
achieving strong recall for high-risk conditions while maintaining
interpretability and conservative error behavior.
